# tools.browse

> Fetch a URL and extract its readable content as markdown, so the model can decide *after* seeing search snippets that it wants to read the full page.

In [ ]:
#| default_exp tools.browse

In [ ]:
#| hide
from nbdev.showdoc import *

## The tool

`trafilatura` handles both fetch and readable-content extraction in a single library. It strips boilerplate (nav, ads, footers), preserves headings/lists/code/tables, and outputs markdown — which is exactly the shape we want to feed back to the model.

Truncation is on by default. Pages are unpredictably large; a paywall warning is fine to lose, a 200KB longform article would otherwise blow context budget unannounced. We surface the truncation explicitly so the model knows there's more if it needs to call again with a higher `max_chars`.

We deliberately catch fetch/extract failures and return them as readable strings rather than raising. That keeps the tool loop alive — the model sees the error, can recover (try a different URL, fall back to search snippets), and the user still gets a final answer.

In [ ]:
#| export
import trafilatura
from nbdialog.core import Tool

def _browse(url: str, max_chars: int = 10000) -> str:
    "Fetch `url` and return its main readable content as markdown, truncated to `max_chars`."
    html = trafilatura.fetch_url(url)
    if not html: return f"(fetch failed for {url})"
    extracted = trafilatura.extract(html, output_format="markdown",
                                    include_links=True, include_tables=True)
    if not extracted: return f"(no readable content extracted from {url})"
    if len(extracted) > max_chars:
        return extracted[:max_chars] + f"\n\n…(truncated; original {len(extracted)} chars — call again with a higher `max_chars` to read more)"
    return extracted

The `Tool` pairs that function with the schema the model sees — the description is what tells it to reach for `browse` after a `web_search` result looks worth reading in full.

In [ ]:
#| export
browse = Tool(
    schema={
        "type": "function",
        "function": {
            "name": "browse",
            "description": "Fetch a single URL and return its main readable content as markdown. Use this after `web_search` when a result looks promising and you need the full page text, not just the snippet.",
            "parameters": {
                "type": "object",
                "properties": {
                    "url": {"type": "string", "description": "The URL to fetch, including scheme (https://...)."},
                    "max_chars": {"type": "integer", "description": "Cap on returned characters (default 10000). Raise this if you got a truncation marker and want to read more.", "default": 10000},
                },
                "required": ["url"],
            },
        },
    },
    fn=_browse,
)

In [ ]:
browse.schema['function']['name'], list(browse.schema['function']['parameters']['properties'])

('browse', ['url', 'max_chars'])

Smoke test — works offline-free if the network is reachable:

In [ ]:
print(_browse("https://example.com", max_chars=400))

This domain is for use in documentation examples without needing permission. Avoid use in operations.

Learn more


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()